# Forest loss drivers in Guinea

1. Forest Loss Definition
2. Current implemented predictors
3. Missing predictors from literature
4. Guinea's forest loss and per ecoregion
5. Test with WRI / Google DeepMind driver map

## 1. Definition

**Forest loss**: loss of tree cover above 30 % canopy, as mapped by Global Forest Change (Hansen et al. 2013, v1.13),
from 2013 to 2025, the years made with the same method (`hansen_annual_maps.ipynb`). Global Forest Watch calls the same
layer *tree cover loss*. It includes deforestation, degradation, clearing within shifting cultivation, and fire.

The official deforestation (NERF 2026: forest → non-forest under the national definition, ~640 km²/yr) is about three times smaller (`lulc_benchmark.ipynb`, 8.8).

**Predictor**: a variable that tells *where* forest loss is likely; the risk model ranks them by importance (association).

**Driver**: *why* forest is lost. Proximate drivers are the activities on the ground (agriculture, wood extraction, infrastructure); underlying drivers are the forces behind them (population, economy, policy) (Geist & Lambin 2002).

This work models **predictors**. Drivers are used to choose the predictors and to explain the causes of the loss.

## 2. The predictors of the current risk model

The current risk model uses 13 predictors. Two things decide whether a predictor can be used:

- **When it is measured.** The yearly layers (land-surface temperature, CHIRPS rainfall, ERA5-Land soil moisture, WorldPop) are loaded at their **latest** year. A cause has to be measured before its effect.
- **Distance to forest edge.** The model is calibrated with the edges of the forest at the **end** of the calibration window, partly the edges created by the loss being predicted, and tested with the forest at the **start** of the test window. 

| Predictor | Importance | Type | Action | Why | Reference |
|---|---|---|---|---|---|
| Distance to forest edge | 0.62 | Cause | Fix: take the edges from the forest at the start of the period, in both calibration and test | Loss happens at forest edges, but the edges were taken after the loss | [Verra VT0007](https://verra.org/methodologies/vt0007-unplanned-deforestation-allocation-tool-udef-at/) |
| Land-surface temperature | 0.09 | Effect | Drop | Clearing heats the ground, and it is measured after the loss | [Li 2015](https://doi.org/10.1038/ncomms7603), [Alkama 2016](https://doi.org/10.1126/science.aac8083) |
| Distance to prior loss | 0.07 | Cause | Keep | New clearing starts next to old clearing | [Soares-Filho 2002](https://doi.org/10.1016/S0304-3800(02)00059-5) |
| Palm-oil price | 0.04 | Weak | Replace with crop suitability | One year of prices from a few markets | [Busch 2017](https://doi.org/10.1093/reep/rew013) |
| Distance to railways | 0.03 | Cause | Keep | The railways serve the mines | [Sonter 2017](https://doi.org/10.1038/s41467-017-00557-w) |
| Distance to roads | 0.02 | Cause | Fix: use the roads that existed at the start of the period (GRIP4, ~2018) | Roads open forest to clearing; today's map has roads built after the loss | [Busch 2017](https://doi.org/10.1093/reep/rew013), [Pfaff 2007](https://doi.org/10.1111/j.1467-9787.2007.00502.x) |
| Elevation | 0.02 | Cause | Keep | High land is harder to clear and farm | [Busch 2017](https://doi.org/10.1093/reep/rew013) |
| Population density | 0.02 | Cause | Fix: use the population of the start year | More people, more clearing; measured after the loss | [Busch 2017](https://doi.org/10.1093/reep/rew013), [Stevens 2015](https://doi.org/10.1371/journal.pone.0107042) |
| Soil moisture | 0.02 | Weak | Drop | One year of weather; repeats rainfall | [Muñoz-Sabater 2021](https://doi.org/10.5194/essd-13-4349-2021), [Dormann 2013](https://doi.org/10.1111/j.1600-0587.2012.07348.x) |
| Precipitation | 0.02 | Cause | Fix: use the long-term average (e.g. 1991–2020) | Climate decides what can be farmed; one year after the loss is weather, not climate | [Busch 2017](https://doi.org/10.1093/reep/rew013), [WMO 2017](https://www.ncei.noaa.gov/pub/data/normals/WMO/Guidelines%20for%20the%20Calculation%20of%20Climate%20Normals.WMO%20No1203_en.pdf) |
| Slope | 0.01 | Cause | Keep | Steep land is harder to clear | [Busch 2017](https://doi.org/10.1093/reep/rew013) |
| Points of interest | 0.01 | Effect | Replace with travel time to towns | Shows today's development, after the loss | [Weiss 2018](https://doi.org/10.1038/nature25181) |
| Distance to waterways | 0.01 | Cause | Keep | Rivers give access and transport | [Serneels 2001](https://doi.org/10.1016/S0167-8809(01)00188-8) |
| Aspect | 0.01 | Weak | Drop | No link to forest loss in the tropics | [Holland & Steyn 1975](https://doi.org/10.2307/3037989) |
| NDVI (already rejected) | – | Effect | Drop | It is the vegetation itself | [Tucker 1979](https://doi.org/10.1016/0034-4257(79)90013-0) |
| Burned area (already rejected) | – | Cause | Replace with fire history before the period | Bush fires are an official driver | [NERF 2026, §VIII.b](https://redd.unfccc.int/media/nerf_de_la_guinee_2026.pdf), [Dwomoh 2017](https://doi.org/10.3390/rs9111117) |
| Night lights (already rejected) | – | Cause | Fix: use the start year, then test again | Settlements and economic activity | [Elvidge 2017](https://doi.org/10.1080/01431161.2017.1342050), [Henderson 2012](https://doi.org/10.1257/aer.102.2.994) |

**Handling every predictor**: measure it at, or before, the start of the window it predicts.

**After predictor check**: re-run the methodology's hold-out test with every predictor measured at the window start, including
distance to forest edge.

## 3. Missing drivers

Drivers named by the project's concept note, by Guinea's REDD+ reference level (NERF 2026, section VIII.b) or by the literature, with no predictor in the model:

| Missing predictor | Driver it stands for | Asked for / named by | Literature reference | Candidate data |
|---|---|---|---|---|
| Protected areas and classified forests | Protection (lowers loss) | Concept note (scenario modelling: protected areas) | Andam et al. 2008; Busch & Ferretti-Gallon 2017 | WDPA (`WCMC/WDPA/current/polygons`), already in the land-use module |
| Land zoning and development plans | Policy (scenarios) | Concept note (scenario modelling: land zoning regulations, development plans) | Soares-Filho et al. 2006 | National plans (to request) |
| Mining permits and sites | Industrial and artisanal mining | NERF 2026 (*activités minières*, *orpaillage*) | Sonter et al. 2017; Giljum et al. 2022; Sims et al. 2025 | National mining cadastre (to request); global mining areas (Maus et al. 2022) |
| Cropland at the window start (distance to it) | Agricultural expansion, the main driver | NERF 2026 (*extension agricole*); concept note (historical LU/LC data, crop management) | Geist & Lambin 2002 (agricultural expansion in 96 % of the cases); Fitzgerald et al. 2021 | Land cover product |
| Crop suitability | Returns to farming | Concept note (crop management) | Busch & Ferretti-Gallon 2017 | FAO GAEZ v4 (Fischer et al. 2021) |
| Travel time to towns and markets | Fuelwood and charcoal demand, market access | NERF 2026 (*bois énergie*); concept note (urban expansion) | Chomitz & Gray 1996; Ahrends et al. 2010; Busch & Ferretti-Gallon 2017 | Weiss et al. 2018 (`Oxford/MAP/accessibility_to_cities_2015_v1_0`) |
| Road network at the window start | Infrastructure | Concept note (infrastructure) | Geist & Lambin 2002; Laurance et al. 2009; Pfaff et al. 2007 | GRIP4 (Meijer et al. 2018), a single ~2018 snapshot; or OpenStreetMap as it was at the window start (ohsome; Raifer et al. 2019). Neither has build dates: OSM history only tells when a road was mapped |
| Built-up area and its growth | Settlements, urban expansion | Concept note (urban expansion); NERF 2026 (*extension urbaine*) | DeFries et al. 2010; Sims et al. 2025 | GHSL built-up surface (`JRC/GHSL/P2023A/GHS_BUILT_S`, every 5 years) |
| Fire history before the window | Bush fires | NERF 2026 (*feux de brousse*) | Dwomoh & Wimberly 2017 | MODIS burned area (`MODIS/061/MCD64A1`) |
| Climate projections | Climate (scenarios only) | Concept note (CMIP6 projections) | Eyring et al. 2016; Thrasher et al. 2022 | NEX-GDDP-CMIP6 (`NASA/GDDP-CMIP6`): for scenarios, not for the risk model |
| Poverty, tenure, governance | Underlying drivers | NERF 2026 (*pauvreté rurale*, *gouvernance*) | Geist & Lambin 2002; Busch & Ferretti-Gallon 2017 (poverty associated with *less* loss, tenure not consistently associated) | Not mappable at 30 m; context per prefecture |

**After predictors check:** Add these predictors, measured at the window start, and keep each one only if it improves the hold-out on every split.

## 4. Drivers of forest loss in Guinea

### 4.1 National, official

Guinea's REDD+ reference level (NERF 2026, section VIII.b): deforestation and degradation are *"principalement dues à
l'extension agricole, à l'exploitation non durable du bois énergie, aux feux de brousse, aux activités minières et à
l'exploitation forestière informelle"*, reinforced by population growth, rural poverty and weak forest governance. In its
1,688 sample plots, deforestation is most often conversion to rain-fed crops, then to grassland; degradation is mostly
caused by agriculture, then logging (NERF 2026, p. 45).

### 4.2 By ecoregion

Loss: Hansen, canopy > 30 %, mean 2013–2025.

**Evidence** says how well the sources *measure* the drivers in that ecoregion:
- *strong*: a peer-reviewed study in Guinea that measured the drivers on a sample;
- *medium*: peer-reviewed, but one small area, or not a measurement of drivers;
- *weak*: no study measured them, only grey literature: reports, news stories, project pages and conference abstracts,
  which are not peer-reviewed.

The last column is the WRI / DeepMind result for 2013–2024 (section 5).

| Ecoregion | Loss (km²/yr, share) | Likely main drivers | Sources | Evidence | WRI / DeepMind |
|---|---|---|---|---|---|
| Guinean forest-savanna | 1,056 (64 %) | Shifting cultivation and fallow cycles, bush fires, fuelwood and charcoal. Around Kissidougou, forest islands are partly *made* by villagers, so part of the "loss" sits inside a fallow cycle | NERF 2026; Fairhead & Leach 1996 | Medium | 84 % shifting cultivation, 15 % permanent agriculture |
| Western Guinean lowland forests | 352 (21 %) | Smallholder agriculture (subsistence and cash crops); logging (degradation); mining nearby (Simandou, Nimba) | Fitzgerald et al. 2021 (Guinée Forestière: 10,907 km² of tree cover loss 2000–2018, agriculture the primary driver) | Strong | 82 % shifting cultivation, 17 % permanent agriculture |
| Guinean montane forests | 197 (12 %) | Fouta Djallon: farming and fires set for pasture and clearing. Nimba and Simandou: iron-ore mining | NASA Earth Observatory 2004; UNEP 2017; NERF 2026 | Weak | 83 % shifting cultivation, 16 % permanent agriculture, 1 % logging |
| West Sudanian savanna | 28 (2 %) | Agricultural expansion (63 % of the loss of 20 sacred forests, 1986–2017), artisanal gold mining, fire, wood cutting | Soumah et al. 2022; AIJR 2022 (Kankan) | Weak to medium | 91 % permanent agriculture, 9 % shifting cultivation |
| Guinean mangroves | 27 (2 %) | Conversion to rice paddies, fuelwood for salt and fish smoking, demand from Conakry | Balde et al. 2014 (Dubréka: 41.7 % of the area changed); Rufford project, Forecariah | Medium | 86 % shifting cultivation, 12 % permanent agriculture |

Check this table against data, ecoregion by ecoregion.

## 5. Drivers mapped by WRI / Google DeepMind

### 5.1 The dataset

*Global drivers of forest loss at 1 km* (Sims et al. 2025), version 1.2, 2001–2024, on Earth Engine. A neural network,
trained on ~7,000 interpreted samples, gives every 1 km cell with tree cover loss its **dominant** driver over the whole
period. It attributes Hansen's loss, the same layer we use.

- **Accuracy**: 90.5 ± 1.4 % globally, **87.4 ± 4.2 % in Africa** (Sims et al. 2025).
- **Limits**: one driver per 1 km cell and per 24 years, so mixed or successive drivers are not separated; natural
  forest and tree plantations are not distinguished.
- **Agriculture is split in two**, which matters in Guinea: *permanent agriculture* includes seasonal crops with a
  fallow period, as long as the land stays farmed; *shifting cultivation* is clearing later abandoned to regrowth (Curtis et al. 2018).

| Literature / NERF driver | WRI / DeepMind class |
|---|---|
| Agricultural expansion | Permanent agriculture, Shifting cultivation |
| Industrial and artisanal mining | Hard commodities |
| Logging | Logging (stand-replacing only; selective logging is degradation and rarely Hansen loss) |
| Bush fires | Wildfire (loss from fire not followed by another use) |
| Urban expansion, roads | Settlements and infrastructure |
| Fuelwood and charcoal | No class of its own: falls under agriculture or logging |

This section computes, for each ecoregion, how much of the Hansen loss of 2013–2024 falls in cells of each driver.

### 5.2 Setup

In [10]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent                     # .../Guinea_Methodology

KEY_CANDIDATES = [
    PROJECT_ROOT / "gee-account.json",
    PROJECT_ROOT / "Land Use-Deforestation" / "landuse_tool" / "credentials" / "tribal-mapper-275801-7b57ce84f3bd.json",
]
KEY_FILE = next((path for path in KEY_CANDIDATES if path.exists()), None)

OUTPUT_DIR = Path.cwd() / "output_datasets"
ECOREGIONS_GPKG = OUTPUT_DIR / "ecoregions_guinea.gpkg"   # Guinea boundary and ecoregions (ecoregions.ipynb)
DRIVERS_PARQUET = OUTPUT_DIR / "forest_loss_by_driver_ecoregion_year.parquet"
DRIVERS_RUN_JSON = OUTPUT_DIR / "forest_loss_by_driver_ecoregion_year.run.json"

In [11]:
HANSEN_ASSET = "UMD/hansen/global_forest_change_2025_v1_13"
DRIVERS_ASSET = "projects/landandcarbon/assets/wri_gdm_drivers_forest_loss_1km/v1_2_2001_2024"  # Sims et al. 2025
CANOPY_THRESHOLD = 30              # % canopy in 2000, as in deforestation_hansen.ipynb and the platform
FIRST_YEAR, LAST_YEAR = 2013, 2024  # Hansen is consistent from 2013; the driver map ends in 2024
FORCE_RECOMPUTE = False            # True = call Earth Engine again even if the parquet already exists

# Driver classes and colours, from the dataset's catalogue page; 0 = loss in a cell the map gives no driver
DRIVER_CLASSES = {
    1: ("Permanent agriculture", "#E39D29"), 2: ("Hard commodities", "#E58074"), 3: ("Shifting cultivation", "#E9D700"),
    4: ("Logging", "#51A44E"), 5: ("Wildfire", "#895128"), 6: ("Settlements and infrastructure", "#A354A0"),
    7: ("Other natural disturbances", "#3A209A"), 0: ("No driver mapped", "#BDBDBD"),
}
DRIVER_NAMES = {code: name for code, (name, _) in DRIVER_CLASSES.items()}
DRIVER_COLORS = {name: color for name, color in DRIVER_CLASSES.values()}

In [12]:
with open(KEY_FILE) as f:
    key_info = json.load(f)

SERVICE_ACCOUNT = key_info["client_email"]
GEE_PROJECT = key_info["project_id"]

credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, key_file=str(KEY_FILE))
ee.Initialize(credentials=credentials, project=GEE_PROJECT)

guinea_boundary = gpd.read_file(ECOREGIONS_GPKG, layer="guinea_boundary")
guinea_ee = ee.Geometry(guinea_boundary.geometry.iloc[0].simplify(0.001).__geo_interface__)  # ~100 m simplification

ecoregions = gpd.read_file(ECOREGIONS_GPKG, layer="ecoregions_guinea")  # from ecoregions.ipynb
ECOREGION_INDEX = {name: index for index, name in enumerate(sorted(ecoregions["ECO_NAME"]), start=1)}  # 0 = outside
ecoregions_ee = ee.FeatureCollection(
    [ee.Feature(ee.Geometry(geometry.simplify(0.002).__geo_interface__), {"eco_index": ECOREGION_INDEX[name]})
     for name, geometry in zip(ecoregions["ECO_NAME"], ecoregions.geometry)]
)
ecoregion_borders = ecoregions_ee.style(color="ffffff", fillColor="00000000", width=1)

### 5.3 Loss per driver, ecoregion and year (Earth Engine)

In [13]:
def get_info_with_eecu(request):
    # Compute an Earth Engine object and return (result, seconds, eecu_seconds); as in hansen_annual_maps.ipynb
    profile_ids = []
    start = time.perf_counter()
    with ee.data.profiling(profile_ids.append):
        result = request.getInfo()
    seconds = time.perf_counter() - start

    get_profiles = ee.ApiFunction.lookup("Profile.getProfiles").call
    for attempt in range(5):
        try:
            profile_text = get_profiles(ids=profile_ids).getInfo()
        except ee.EEException:
            time.sleep(2 * (attempt + 1))
            continue
        eecu_seconds = sum(
            float(line.split()[0]) for line in profile_text.splitlines()[1:] if line.split() and line.split()[0] != "-"
        )
        return result, seconds, round(eecu_seconds, 1)
    return result, seconds, None


def loss_by_driver_ecoregion_year():
    hansen = ee.Image(HANSEN_ASSET)
    projection = hansen.select("lossyear").projection().getInfo()
    ecoregion_image = ee.Image(0).int().paint(ecoregions_ee, "eco_index")
    driver = ee.Image(DRIVERS_ASSET).select("classification").unmask(0)
    code = ecoregion_image.multiply(10000).add(driver.multiply(100)).add(hansen.select("lossyear")).toInt()
    is_loss = (
        hansen.select("lossyear").gt(0)
        .And(hansen.select("treecover2000").gt(CANOPY_THRESHOLD))
        .And(hansen.select("datamask").eq(1))
    )
    request = (
        ee.Image.pixelArea().divide(1e6).addBands(code).updateMask(is_loss)
        .reduceRegion(
            reducer=ee.Reducer.sum().unweighted().group(groupField=1, groupName="code"),
            geometry=guinea_ee, crs=projection["crs"], crsTransform=projection["transform"], maxPixels=1e13,
        )
    )
    result, seconds, eecu_seconds = get_info_with_eecu(request)
    table = pd.DataFrame(result["groups"]).rename(columns={"sum": "loss_km2"})
    table["eco_index"] = table["code"] // 10000
    table["driver"] = table["code"] // 100 % 100
    table["year"] = 2000 + table["code"] % 100
    return table.drop(columns="code"), seconds, eecu_seconds


if DRIVERS_PARQUET.exists() and not FORCE_RECOMPUTE:
    drivers_table = pd.read_parquet(DRIVERS_PARQUET)
    print(f"Read {DRIVERS_PARQUET.name} (no Earth Engine compute used)")
else:
    drivers_table, seconds, eecu_seconds = loss_by_driver_ecoregion_year()
    drivers_table.to_parquet(DRIVERS_PARQUET, index=False)
    DRIVERS_RUN_JSON.write_text(json.dumps({
        "computed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "hansen_asset": HANSEN_ASSET, "drivers_asset": DRIVERS_ASSET, "canopy_threshold": CANOPY_THRESHOLD,
        "seconds": round(seconds, 1), "eecu_seconds": eecu_seconds,
        "eecu_hours": None if eecu_seconds is None else round(eecu_seconds / 3600, 3),
    }, indent=2))
    print(f"Computed in {seconds:.0f} s, {eecu_seconds} EECU-seconds")

ECOREGION_NAMES = {index: name for name, index in ECOREGION_INDEX.items()} | {0: "(outside the ecoregion polygons)"}
drivers_table["ECO_NAME"] = drivers_table["eco_index"].map(ECOREGION_NAMES)
drivers_table["driver_name"] = drivers_table["driver"].map(DRIVER_NAMES)
window = drivers_table[drivers_table["year"].between(FIRST_YEAR, LAST_YEAR)]
print(f"Loss {FIRST_YEAR}–{LAST_YEAR}: {window['loss_km2'].sum():,.0f} km² "
      f"({window['loss_km2'].sum() / (LAST_YEAR - FIRST_YEAR + 1):,.0f} km²/yr)")

Read forest_loss_by_driver_ecoregion_year.parquet (no Earth Engine compute used)
Loss 2013–2024: 20,510 km² (1,709 km²/yr)


### 5.4 Drivers per ecoregion

Mean loss per year in `FIRST_YEAR`–`LAST_YEAR`, split by the driver of its 1 km cell. Left as km²/yr, right as a share
of each ecoregion's loss.

In [14]:
N_YEARS = LAST_YEAR - FIRST_YEAR + 1
driver_order = [name for name, _ in DRIVER_CLASSES.values()]

per_ecoregion = window.groupby(["ECO_NAME", "driver_name"])["loss_km2"].sum().unstack(fill_value=0) / N_YEARS
per_ecoregion.loc["Guinea"] = per_ecoregion.sum()
per_ecoregion = per_ecoregion.reindex(columns=[name for name in driver_order if name in per_ecoregion.columns])
per_ecoregion = per_ecoregion.loc[per_ecoregion.sum(axis=1).sort_values().index]   # largest loss on top of the chart
share = 100 * per_ecoregion.div(per_ecoregion.sum(axis=1), axis=0)

for table, value, title in [
    (per_ecoregion, "Loss (km²/yr)", f"Forest loss per ecoregion and driver, mean {FIRST_YEAR}–{LAST_YEAR}"),
    (share, "Share of the ecoregion's loss (%)", "Share of each driver in the ecoregion's loss"),
]:
    long = table.reset_index().melt(id_vars="ECO_NAME", var_name="Driver", value_name=value)
    fig = px.bar(long, y="ECO_NAME", x=value, color="Driver", orientation="h", color_discrete_map=DRIVER_COLORS,
                 category_orders={"Driver": driver_order}, labels={"ECO_NAME": ""}, title=title,
                 template="plotly_white", height=380)
    fig.show()

per_ecoregion.insert(0, "total_km2_per_year", per_ecoregion.sum(axis=1))
display(per_ecoregion.iloc[::-1].round(1))
share.iloc[::-1].round(1)

driver_name,total_km2_per_year,Permanent agriculture,Hard commodities,Shifting cultivation,Logging,Wildfire,Settlements and infrastructure,Other natural disturbances,No driver mapped
ECO_NAME,,,,,,,,,
Guinea,1709.2,280.9,10.1,1405.5,7.0,0.1,3.7,1.5,0.3
Guinean forest-savanna,1092.0,160.9,9.0,916.9,3.6,0.1,0.8,0.6,0.2
Western Guinean lowland forests,360.0,61.5,0.5,293.9,0.5,0.1,2.7,0.7,0.1
Guinean montane forests,203.8,32.2,0.4,168.2,2.8,0.0,0.1,0.0,0.1
Guinean mangroves,27.8,3.4,0.2,23.9,0.1,0.0,0.1,0.2,0.0
West Sudanian savanna,25.2,22.8,0.0,2.2,0.0,0.0,0.0,0.0,0.0
(outside the ecoregion polygons),0.4,0.1,0.0,0.3,0.0,0.0,0.0,0.0,0.0


driver_name,Permanent agriculture,Hard commodities,Shifting cultivation,Logging,Wildfire,Settlements and infrastructure,Other natural disturbances,No driver mapped
ECO_NAME,,,,,,,,
Guinea,16.4,0.6,82.2,0.4,0.0,0.2,0.1,0.0
Guinean forest-savanna,14.7,0.8,84.0,0.3,0.0,0.1,0.1,0.0
Western Guinean lowland forests,17.1,0.1,81.6,0.1,0.0,0.8,0.2,0.0
Guinean montane forests,15.8,0.2,82.6,1.4,0.0,0.1,0.0,0.0
Guinean mangroves,12.2,0.8,85.9,0.2,0.0,0.3,0.6,0.0
West Sudanian savanna,90.7,0.1,8.9,0.1,0.0,0.0,0.0,0.1
(outside the ecoregion polygons),26.0,0.8,70.3,0.0,0.0,0.1,2.7,0.1


### 5.5 Drivers per year

National loss per year and driver. The map has no yearly drivers: each 1 km cell has one driver for all of 2001–2024.
Each year's Hansen loss is labelled with the driver of the cell it falls in. So what changes from year to year is how
much is lost, not the driver. A cell that went from farming to mining keeps one driver for every year.

In [15]:
per_year = window.groupby(["year", "driver_name"])["loss_km2"].sum().reset_index()
fig = px.bar(per_year, x="year", y="loss_km2", color="driver_name", color_discrete_map=DRIVER_COLORS,
             category_orders={"driver_name": driver_order},
             labels={"loss_km2": "Loss (km²)", "year": "Year", "driver_name": "Driver"},
             title="Forest loss per year, by driver of its 1 km cell", template="plotly_white", height=380)
fig.show()
per_year.pivot(index="year", columns="driver_name", values="loss_km2").reindex(
    columns=[name for name in driver_order if name in set(per_year["driver_name"])]).round(0)

driver_name,Permanent agriculture,Hard commodities,Shifting cultivation,Logging,Wildfire,Settlements and infrastructure,Other natural disturbances,No driver mapped
year,,,,,,,,
2013,260.0,7.0,1194.0,8.0,0.0,3.0,1.0,1.0
2014,203.0,6.0,1115.0,5.0,0.0,2.0,1.0,1.0
2015,219.0,6.0,1385.0,12.0,0.0,4.0,1.0,0.0
2016,273.0,12.0,1875.0,6.0,0.0,5.0,3.0,0.0
2017,347.0,11.0,1953.0,5.0,0.0,6.0,2.0,0.0
2018,284.0,9.0,1530.0,2.0,NaN,4.0,3.0,0.0
2019,314.0,10.0,1606.0,9.0,0.0,4.0,2.0,0.0
2020,291.0,8.0,1466.0,23.0,1.0,5.0,1.0,0.0
2021,268.0,9.0,1392.0,6.0,0.0,4.0,2.0,0.0


### 5.6 Map

The dominant driver of every 1 km cell, over the Hansen loss of `FIRST_YEAR`–`LAST_YEAR` and the ecoregion borders.

In [16]:
hansen = ee.Image(HANSEN_ASSET)
recent_loss = (
    hansen.select("lossyear").gte(FIRST_YEAR - 2000).And(hansen.select("lossyear").lte(LAST_YEAR - 2000))
    .And(hansen.select("treecover2000").gt(CANOPY_THRESHOLD))
)
driver_map = ee.Image(DRIVERS_ASSET).select("classification")
mapped_classes = [code for code in DRIVER_CLASSES if code > 0]

m = geemap.Map(center=[10.4, -11], zoom=7, height="650px")
m.add_basemap("SATELLITE")
m.add_layer(driver_map.clip(guinea_ee), {"min": 1, "max": 7, "palette": [DRIVER_CLASSES[code][1] for code in mapped_classes]},
            "Driver (WRI / DeepMind, 1 km)", True, 0.7)
m.add_layer(recent_loss.selfMask().clip(guinea_ee), {"palette": ["ff0000"]}, f"Forest loss {FIRST_YEAR}–{LAST_YEAR}", False)
m.add_layer(ecoregion_borders, {}, "Ecoregion borders")

# Ecoregion names: one label inside each part larger than 2,000 km²
ecoregion_parts = ecoregions.explode(index_parts=False)
ecoregion_parts = ecoregion_parts[ecoregion_parts.to_crs(ecoregion_parts.estimate_utm_crs()).area > 2e9]
label_points = ecoregion_parts.representative_point()
m.add_labels(
    pd.DataFrame({"ECO_NAME": ecoregion_parts["ECO_NAME"].to_numpy(),
                  "longitude": label_points.x.to_numpy(), "latitude": label_points.y.to_numpy()}),
    "ECO_NAME", font_size="10pt", font_color="white", font_weight="bold", draggable=False, layer_name="Ecoregion names",
)
m.add_legend(title="Driver", legend_dict={DRIVER_CLASSES[code][0]: DRIVER_CLASSES[code][1] for code in mapped_classes})
m

Map(center=[10.4, -11], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

### 5.7 Findings

Forest loss 2013–2024: 1,709 km²/yr (canopy > 30 %).

1. **Shifting cultivation drives 82 % of forest loss**, and 82–86 % in each forest ecoregion (forest-savanna, lowland, montane, mangroves). Permanent agriculture is most of the rest (16 %). This agrees with the NERF (agricultural expansion, rain-fed crops) and with Fitzgerald et al. (2021).
2. **The West Sudanian savanna is the exception**: 91 % permanent agriculture. It holds only 1.5 % of the loss (25 km²/yr).
3. **Mining, logging, settlements and fire are each under 1 %** (hard commodities 10 km²/yr, logging 7, settlements 4, wildfire ~0). That is under 1 % of the clearing Hansen sees, not of what happens:
   - **Mining**: each 1 km cell (100 ha) gets the one driver behind most of its loss. A cell with a 20 ha mine pit and 50 ha of new fields is labelled agriculture, and the mine's 20 ha count as agriculture too.
   - **Logging**: selective logging takes a few trees and leaves the canopy. Hansen only records loss when a 30 m pixel is cleared, so most logging is not in it.
   - **Fire**: savanna fires burn the grass, not the trees, so there is no Hansen loss. Fire used to clear a field is counted as agriculture.
4. **Over time**, shifting cultivation peaks in 2016–2017 (~1,900 km²) and falls to ~1,130 km² in 2024, while permanent agriculture rises slowly (260 → 330 km²) and hard commodities double (7 → 14–18 km²), still small.
5. **Much of this loss grows back.** Shifting cultivation clears a plot, farms it for a few years, then leaves it to regrow (Sims et al. 2025). Hansen counts every clearing as loss and never subtracts the regrowth: its regrowth layer stops in 2012, and each pixel has only one loss year. Official deforestation counts only forest that stays non-forest. So with regrowth, net deforestation is smaller. That is why Hansen's loss is about three times the official figure, and why we call it forest loss.

**What it means for the predictors**

- The same process dominates every forest ecoregion, so the drivers do not call for a separate model per ecoregion:
  one national model, with the ecoregion as a predictor and results reported per ecoregion. The savanna has a different
  driver but too little loss to model on its own.
- The predictors should describe **where farmers clear next**: distance to recent clearing and to the forest edge (both
  at the window start), distance to settlements and travel time to towns, population, cropland and fallow at the
  window start, slope.
- Mining, roads and railways explain little of the loss nationally. Keep mining as a scenario layer (concession areas)
  rather than a main risk predictor.
- **Limits**: one driver per 1 km cell for 2001–2024, 87 % accuracy in Africa, and an uncertain line between shifting
  cultivation and permanent agriculture where farming uses fallows. To confirm, label a sample of loss points on
  high-resolution imagery.
- **Regrowth is not in the model.** The model's forest is the 2000 forest minus the loss, so forest that grew back after
  2000 counts as non-forest. In a fallow system that is the next forest to be cleared, and it also moves the forest
  edges. Regrowth counts as tree cover again only once the trees reach 5 m (Hansen's definition). We found no figure
  for fallow length in Guinea; West African fallows have become shorter over time. Both can be measured (section 6).


## 6. Next steps

1. **Re-test the current predictors**, all measured at the window start, and distance to forest edge computed from the
   forest at the window start, with the methodology's hold-out test, per ecoregion as well as nationally.
2. **Add the missing predictors** of section 3, each kept only if it improves the hold-out on every split.
3. **Regrowth**: measure how much of the 2013–2016 loss is tree cover again in a recent land-cover map, and use a recent
   forest map (e.g. 2020) instead of the 2000 forest as the forest at the window start.

## References

- AIJR (2022). *Impact of Artisanal Gold Mining on the Populations of Upper Guinea, Administrative Region of Kankan*.
  AIJR Proceedings. https://books.aijr.org/index.php/press/catalog/book/139/chapter/2175
- Ahrends, A., Burgess, N. D., Milledge, S. A. H., et al. (2010). Predictable waves of sequential forest degradation and
  biodiversity loss spreading from an African city. *PNAS*, 107(33), 14556–14561.
- Alkama, R., & Cescatti, A. (2016). Biophysical climate impacts of recent changes in global forest cover. *Science*,
  351(6273), 600–604.
- Andam, K. S., Ferraro, P. J., Pfaff, A., Sanchez-Azofeifa, G. A., & Robalino, J. A. (2008). Measuring the
  effectiveness of protected area networks in reducing deforestation. *PNAS*, 105(42), 16089–16094.
- Balde, B. S., Kobayashi, H., Matsumura, I., et al. (2014). Land use change and their determinants in the coastal area
  of Guinea: a study based on spatial analysis and field survey. *Journal of the Japanese Agricultural Systems
  Society*, 30(3), 66–75.
- Busch, J., & Ferretti-Gallon, K. (2017). What drives deforestation and what stops it? A meta-analysis. *Review of
  Environmental Economics and Policy*, 11(1), 3–23.
- Chomitz, K. M., & Gray, D. A. (1996). Roads, land use, and deforestation: a spatial model applied to Belize. *The
  World Bank Economic Review*, 10(3), 487–512.
- Curtis, P. G., Slay, C. M., Harris, N. L., Tyukavina, A., & Hansen, M. C. (2018). Classifying drivers of global
  forest loss. *Science*, 361(6407), 1108–1111.
- DeFries, R. S., Rudel, T., Uriarte, M., & Hansen, M. (2010). Deforestation driven by urban population growth and
  agricultural trade in the twenty-first century. *Nature Geoscience*, 3, 178–181.
- Dormann, C. F., Elith, J., Bacher, S., et al. (2013). Collinearity: a review of methods to deal with it and a
  simulation study evaluating their performance. *Ecography*, 36(1), 27–46.
- Dwomoh, F. K., & Wimberly, M. C. (2017). Fire regimes and their drivers in the Upper Guinean Region of West Africa.
  *Remote Sensing*, 9(11), 1117.
- Elvidge, C. D., Baugh, K., Zhizhin, M., Hsu, F. C., & Ghosh, T. (2017). VIIRS night-time lights. *International
  Journal of Remote Sensing*, 38(21), 5860–5879.
- Eyring, V., et al. (2016). Overview of the Coupled Model Intercomparison Project Phase 6 (CMIP6) experimental design
  and organization. *Geoscientific Model Development*, 9, 1937–1958.
- Fairhead, J., & Leach, M. (1996). *Misreading the African Landscape: Society and Ecology in a Forest-Savanna Mosaic*.
  Cambridge University Press.
- Fischer, G., et al. (2021). *Global Agro-Ecological Zones v4: Model documentation*. Rome: FAO and IIASA.
- Fitzgerald, M., Nackoney, J., Potapov, P., & Turubanova, S. (2021). Agriculture is the primary driver of tree cover
  loss across the Forestière region of the Republic of Guinea, Africa. *Environmental Research Communications*,
  3(12), 121004.
- Giljum, S., Maus, V., Kuschnig, N., et al. (2022). A pantropical assessment of deforestation caused by industrial
  mining. *PNAS*, 119(38), e2118273119.
- Geist, H. J., & Lambin, E. F. (2002). Proximate causes and underlying driving forces of tropical deforestation.
  *BioScience*, 52(2), 143–150.
- Hansen, M. C., et al. (2013). High-resolution global maps of 21st-century forest cover change. *Science*, 342(6160),
  850–853.
- Henderson, J. V., Storeygard, A., & Weil, D. N. (2012). Measuring economic growth from outer space. *American
  Economic Review*, 102(2), 994–1028.
- Holland, P. G., & Steyn, D. G. (1975). Vegetational responses to latitudinal variations in slope angle and aspect.
  *Journal of Biogeography*, 2(3), 179–183.
- Laurance, W. F., Goosem, M., & Laurance, S. G. W. (2009). Impacts of roads and linear clearings on tropical forests.
  *Trends in Ecology & Evolution*, 24(12), 659–669.
- Li, Y., Zhao, M., Motesharrei, S., Mu, Q., Kalnay, E., & Li, S. (2015). Local cooling and warming effects of forests
  based on satellite observations. *Nature Communications*, 6, 6603.
- Maus, V., et al. (2022). An update on global mining land use. *Scientific Data*, 9, 433.
- Meijer, J. R., Huijbregts, M. A. J., Schotten, K. C. G. J., & Schipper, A. M. (2018). Global patterns of current and
  future road infrastructure. *Environmental Research Letters*, 13(6), 064006.
- Muñoz-Sabater, J., et al. (2021). ERA5-Land: a state-of-the-art global reanalysis dataset for land applications.
  *Earth System Science Data*, 13, 4349–4383.
- NASA Earth Observatory (2004). Fires in Guinea (MODIS, April 2004).
- Pfaff, A., et al. (2007). Road investments, spatial spillovers, and deforestation in the Brazilian Amazon. *Journal
  of Regional Science*, 47(1), 109–123.
- Pontius, R. G., Jr., & Malanson, J. (2005). Comparison of the structure and accuracy of two land change models.
  *International Journal of Geographical Information Science*, 19(2), 243–265.
- Raifer, M., Troilo, R., Kowatsch, F., et al. (2019). OSHDB: a framework for spatio-temporal analysis of OpenStreetMap
  history data. *Open Geospatial Data, Software and Standards*, 4, 3.
- République de Guinée (2026). *Niveau de référence des forêts (NRF) et niveau d'émissions de référence des forêts
  (NERF) de la Guinée, période 2016–2020*. Submitted to the UNFCCC. https://redd.unfccc.int/media/nerf_de_la_guinee_2026.pdf
- Rufford Foundation. Cissé, O. *Understanding external drivers of and community pathways for reducing mangrove logging
  in Guinea* (Forécariah). Project page.
- Serneels, S., & Lambin, E. F. (2001). Proximate causes of land-use change in Narok District, Kenya: a spatial
  statistical model. *Agriculture, Ecosystems & Environment*, 85(1–3), 65–81.
- Sims, M. J., et al. (2025). Global drivers of forest loss at 1 km resolution. *Environmental Research Letters*,
  20(7), 074027.
- Soares-Filho, B. S., Cerqueira, G. C., & Pennachin, C. L. (2002). DINAMICA: a stochastic cellular automata model
  designed to simulate the landscape dynamics in an Amazonian colonization frontier. *Ecological Modelling*, 154(3),
  217–235.
- Soares-Filho, B. S., Nepstad, D. C., Curran, L. M., et al. (2006). Modelling conservation in the Amazon basin.
  *Nature*, 440, 520–523.
- Sonter, L. J., Herrera, D., Barrett, D. J., et al. (2017). Mining drives extensive deforestation in the Brazilian
  Amazon. *Nature Communications*, 8, 1013.
- Soumah, F. S., Kokou, K., Diakité, M., Camara, Y., Kourouma, S., & Kourama, S. (2022). Analyse diachronique grâce aux
  images Landsat de la dynamique spatiale des forêts sacrées du haut bassin du Niger en République de Guinée. *Revue
  Française de Photogrammétrie et de Télédétection*, 223(1), 250–266.
- Stevens, F. R., Gaughan, A. E., Linard, C., & Tatem, A. J. (2015). Disaggregating census data for population mapping
  using random forests with remotely-sensed and ancillary data. *PLoS ONE*, 10(2), e0107042.
- Thrasher, B., Wang, W., Michaelis, A., et al. (2022). NASA Global Daily Downscaled Projections, CMIP6. *Scientific
  Data*, 9, 262.
- Tucker, C. J. (1979). Red and photographic infrared linear combinations for monitoring vegetation. *Remote Sensing
  of Environment*, 8(2), 127–150.
- UNEP (2017). Attitudes to slash and burn in Guinea's highlands are changing. UNEP news story, 13 April 2017.
- Verra (2024). *VT0007 Unplanned Deforestation Allocation (UDef-A)*, v1.0.
- WMO (2017). *WMO Guidelines on the Calculation of Climate Normals* (WMO-No. 1203). Geneva: World Meteorological
  Organization.
- Weiss, D. J., et al. (2018). A global map of travel time to cities to assess inequalities in accessibility in 2015.
  *Nature*, 553, 333–336.